In [2]:
from typing import Optional, Tuple, Literal

import math
from typing import NamedTuple, Optional, Tuple

import numpy as np
import torch
from scipy.sparse import coo_matrix, csr_matrix

import torch

In [3]:
def splot_by_rows(
    src_indices: torch.Tensor, dst_indices: torch.Tensor, row_size: int
) -> list[tuple[int, torch.Tensor, torch.Tensor]]:
    """Split the edge index by block rows.

    Args:
        src_indices (torch.Tensor): [E] long.
        dst_indices (torch.Tensor): [E] long.
        row_size (int): Row size.

    Returns:
        list[tuple[int, torch.Tensor, torch.Tensor]]: List of (row_id, src_indices, dst_indices).
    """
    splitted = src_indices.clone() // row_size
    boundaries = torch.cat([torch.tensor([True], device=src_indices.device), splitted[1:] != splitted[:-1]])
    idx = boundaries.nonzero(as_tuple=True)[0]
    idx = torch.cat([idx, torch.tensor([len(splitted)], device=src_indices.device)])
    return [
        (splitted[idx[i]], src_indices[idx[i] : idx[i + 1]], dst_indices[idx[i] : idx[i + 1]])
        for i in range(len(idx) - 1)
    ]


def non_zero_column_ids(
    src_indices_block: torch.Tensor,
    dst_indices_block: torch.Tensor,
    num_nodes: int,
    row_index: int,
    block_row_size: int,
) -> torch.Tensor:
    """Calculate the column remapping for a block of edges.

    Args:
        src_indices_block (torch.Tensor): [E] long.
        dst_indices_block (torch.Tensor): [E] long.
        num_nodes (int): Number of nodes.

    Returns:
        torch.Tensor: Column remapping.
    """

    row_start = row_index * block_row_size
    src_indices_block = src_indices_block.clone() - row_start
    coordinates = src_indices_block * num_nodes + dst_indices_block
    column_index = coordinates / block_row_size

    column_remapping = torch.unique(column_index)
    return column_remapping


def to_dense_matrix(
    src_indices: torch.Tensor, dst_indices: torch.Tensor, row_index: int, num_nodes: int, block_row_size: int
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Convert CSR to dense matrix.

    Args:
        src_indices (torch.Tensor): [E] long.
        dst_indices (torch.Tensor): [E] long.
        row_index (int): Row index.
        num_nodes (int): Number of nodes.
        block_row_size (int): Block row size.

    Returns:
        Tuple[torch.Tensor, torch.Tensor]: Dense matrix, column remapping.
    """
    non_zero_ids = non_zero_column_ids(src_indices, dst_indices, num_nodes, row_index, block_row_size)
    dense_shape = (block_row_size, non_zero_ids.shape[0])
    dense = torch.zeros(dense_shape, device=src_indices.device).view(-1)
    index_unwrapped = (src_indices - src_indices.min()) * num_nodes + dst_indices
    dense.scatter_(0, index_unwrapped, 1)
    return dense.view(block_row_size, non_zero_ids.shape[0]), non_zero_ids


def to_block_sparse_matrix(
    edge_index: torch.Tensor,
    num_nodes: int,
    edge_weight: Optional[torch.Tensor] = None,
    block_row_size: int = 16,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Create a block sparse matrix lazily.

    Args:
        edge_index (torch.Tensor): [2, E] long.
        num_nodes (int): Number of nodes.
        edge_weight (Optional[torch.Tensor]): Edge weights [E].
        block_row_size (int): Block row size.

    Returns:
        Tuple[torch.Tensor, torch.Tensor, torch.Tensor]: Row pointer, column indices, values.
    """

    src_indices, dst_indices = edge_index[0], edge_index[1]
    blocks = splot_by_rows(src_indices, dst_indices, block_row_size)

    row_block_ids = torch.zeros(block_row_size, device=src_indices.device, dtype=torch.long)

    dense_blocks = []
    column_remappings = []

    for row_id, src_indices_block, dst_indices_block in blocks:
        dense_block, column_remapping = to_dense_matrix(
            src_indices_block, dst_indices_block, row_id, num_nodes, block_row_size
        )
        row_block_ids[row_id] = row_id
        dense_blocks.append(dense_block)
        column_remappings.append(column_remapping)

    dense_block = torch.cat(dense_blocks, dim=1)
    column_remapping = torch.cat(column_remappings, dim=0)

    return row_block_ids, dense_block, column_remapping

In [4]:
ones = torch.ones((16, 16))
zeros = torch.zeros((16, 16))

adj = torch.cat(
    [
        torch.cat([ones, zeros], 1),
        torch.cat([zeros, ones], 1),
    ],
    0,
)

adj[1, 17] = 1
adj

tensor([[1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        [1., 1., 1.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 1., 1., 1.],
        [0., 0., 0.,  ..., 1., 1., 1.]])

In [5]:
edge_index = adj.nonzero().T

In [6]:
def normalize_adj(
    edge_index: torch.Tensor, num_nodes: int, how: Literal["left", "right", "both", "none"], add_self_loops: bool = True
) -> torch.Tensor:
    """Compute symmetric normalized adjacency (A_hat) as sparse COO.

    Args:
        edge_index (torch.Tensor): [2, E] long tensor.
        num_nodes (int): Number of nodes.

    Returns:
        torch.Tensor: Sparse COO adjacency with added self-loops and:
            - D^{-1/2} A D^{-1/2} normalization if `how` == "both".
            - D_in^{-1} A^T normalization if `how` == "right" -- normalization for mean-aggregation
            - A if `how` == "none" -- normalization for adj-mat backend
    """
    device = edge_index.device
    idx = edge_index

    if add_self_loops:
        self_loops = torch.arange(num_nodes, device=device)
        loop_idx = torch.stack([self_loops, self_loops], dim=0)
        idx = torch.cat([idx, loop_idx], dim=1)
        edge_index = idx

    if how == "both":
        values = torch.ones(idx.size(1), device=device)
        adj = torch.sparse_coo_tensor(idx, values, (num_nodes, num_nodes)).T.coalesce().to(values.device)

        deg1 = torch.sparse.sum(adj, dim=1).to_dense()
        D_inv_sqrt1 = torch.pow(deg1.clamp(min=1.0), -0.5)

        deg0 = torch.sparse.sum(adj, dim=0).to_dense()
        D_inv_sqrt0 = torch.pow(deg0.clamp(min=1.0), -0.5)

        idx, values = adj.indices(), adj.values()
        row, col = idx
        norm_vals = D_inv_sqrt1[row] * values * D_inv_sqrt0[col]
        return torch.sparse_coo_tensor(idx, norm_vals, (num_nodes, num_nodes)).coalesce()
    elif how == "left":
        raise NotImplementedError()
    elif how == "right":
        """
            Computes A^T (transposed adjacency) and D_in^{-1} (inverse in-degree diagonal).
            This matches DGL's copy_u_mean operation.
        """
        device = edge_index.device
        src, dst = edge_index[0], edge_index[1]

        values = torch.ones(edge_index.size(1), device=device)
        adj_t_indices = torch.stack([dst, src], dim=0)
        adj_t = torch.sparse_coo_tensor(adj_t_indices, values, (num_nodes, num_nodes)).coalesce()

        in_degrees = torch.zeros(num_nodes, device=device)
        in_degrees.scatter_add_(0, dst, torch.ones_like(dst, dtype=torch.float32))

        # handle isolated nodes (in_degree = 0) by setting to 1 to avoid division by zero
        in_degrees = in_degrees.clamp(min=1.0)

        in_degree_inv = 1.0 / in_degrees
        diag_indices = torch.arange(num_nodes, device=device).unsqueeze(0).repeat(2, 1)
        in_degree_inv_diag = torch.sparse_coo_tensor(diag_indices, in_degree_inv, (num_nodes, num_nodes)).coalesce()

        adj_t_normalized = in_degree_inv_diag @ adj_t
        return adj_t_normalized
    elif how == "none":
        """
            Computes A^T (transposed adjacency).
            This matches DGL's copy_u_sum operation.
        """
        device = edge_index.device
        src, dst = edge_index[0], edge_index[1]

        values = torch.ones(edge_index.size(1), device=device)
        adj_t_indices = torch.stack([dst, src], dim=0)
        adj_t = torch.sparse_coo_tensor(adj_t_indices, values, (num_nodes, num_nodes)).coalesce()

        return adj_t
    else:
        raise ValueError(f"Normalization type {how} is inappropriate")


In [7]:
edge_index = adj.nonzero().T
adjacency_sparse = normalize_adj(edge_index, num_nodes=adj.shape[0], how="both", add_self_loops=False)
adj_csr = adjacency_sparse.to_sparse_csr()

/var/tmp/ipykernel_683467/4227108561.py:3: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at ../aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  adj_csr = adjacency_sparse.to_sparse_csr()


In [8]:
ROW_WINDOW_SIZE = 16  # BLK_M - rows per row window
TCB_WIDTH = 8  # BLK_N - columns per TCB
TCB_SIZE = ROW_WINDOW_SIZE * TCB_WIDTH  # 128 elements per TCB

In [9]:
TCB_SIZE

128

In [10]:
src, dst = adjacency_sparse.indices()

In [11]:
16 * 16 * 2

512

In [12]:
class WSBFormat(NamedTuple):
    """WSB format tensors for CUDA kernel"""

    tcb_row_offset: torch.Tensor  # [num_row_windows + 1], int32
    col_idx: torch.Tensor  # [num_tcbs * 8], int32
    bitmap: torch.Tensor  # [num_tcbs * 2], int64 (uint64)
    weights: torch.Tensor  # [num_tcbs * 128], float16

    num_nodes: int
    num_edges: int
    num_row_windows: int
    num_tcbs: int

    def to(self, device: str) -> "WSBFormat":
        """Move tensors to device"""
        return WSBFormat(
            tcb_row_offset=self.tcb_row_offset.to(device),
            col_idx=self.col_idx.to(device),
            bitmap=self.bitmap.to(device),
            weights=self.weights.to(device),
            num_nodes=self.num_nodes,
            num_edges=self.num_edges,
            num_row_windows=self.num_row_windows,
            num_tcbs=self.num_tcbs,
        )

    def cuda(self) -> "WSBFormat":
        return self.to("cuda")

    def memory_bytes(self) -> int:
        """Total memory footprint"""
        return self.tcb_row_offset.nbytes + self.col_idx.nbytes + self.bitmap.nbytes + self.weights.nbytes

    def __repr__(self) -> str:
        return (
            f"WSBFormat(nodes={self.num_nodes}, edges={self.num_edges}, "
            f"row_windows={self.num_row_windows}, tcbs={self.num_tcbs}, "
            f"memory={self.memory_bytes() / 1024:.1f} KB)"
        )

    def to_dense(self) -> torch.Tensor:
        """
        Convert WSB format back to dense matrix for verification.

        Returns:
        [N, N] dense adjacency matrix
        """
        N = self.num_nodes
        dense = np.zeros((N, N), dtype=np.float32)  # Use numpy array instead

        tcb_row_offset = self.tcb_row_offset.numpy()
        col_idx = self.col_idx.numpy()
        bitmap = self.bitmap.numpy().view(np.uint64)  # view, not astype
        weights = self.weights.float().numpy()

        for rw in range(self.num_row_windows):
            row_start = rw * ROW_WINDOW_SIZE
            tcb_start = tcb_row_offset[rw]
            tcb_end = tcb_row_offset[rw + 1]

            for tcb_idx in range(tcb_start, tcb_end):
                # Get columns for this TCB
                cols = col_idx[tcb_idx * TCB_WIDTH : (tcb_idx + 1) * TCB_WIDTH]

                # Get bitmap
                bm_lo = bitmap[tcb_idx * 2]
                bm_hi = bitmap[tcb_idx * 2 + 1]

                # Get weights
                tcb_weights = weights[tcb_idx * TCB_SIZE : (tcb_idx + 1) * TCB_SIZE]

                # Decode
                for local_row in range(ROW_WINDOW_SIZE):
                    global_row = row_start + local_row
                    if global_row >= N:
                        break

                    for local_col in range(TCB_WIDTH):
                        bit_pos = (local_row % 8) * TCB_WIDTH + local_col
                        bm = bm_lo if local_row < 8 else bm_hi

                        if bm & (np.uint64(1) << np.uint64(bit_pos)):
                            weight_idx = local_row * TCB_WIDTH + local_col
                            global_col = cols[local_col]
                            dense[global_row, global_col] = tcb_weights[weight_idx]

        return torch.from_numpy(dense)


WSB (Weighted Sparse Block) Format Builder for GCN SpMM on Tensor Cores

Based on Fused3S BSB format adapted for precomputed edge weights.

Format:
- Row window: 16 rows (matches m16n8k16 MMA shape)
- TCB (Tensor Core Block): 16 rows × 8 columns = 128 elements
- Bitmap: 128 bits (2 × uint64) per TCB indicating non-zero positions
- Weights: Dense storage within each TCB (128 half values, zeros for non-edges)

Data structures:
- tcb_row_offset[num_row_windows + 1]: Starting TCB index for each row window
- col_idx[total_tcbs * 8]: Column indices for each TCB (8 per TCB)
- bitmap[total_tcbs * 2]: Sparsity bitmaps (2 uint64 per TCB)
- weights[total_tcbs * 128]: Edge weights dense within TCBs


In [13]:
def build_wsb_format(
    adj: torch.Tensor,
    dtype: torch.dtype = torch.float16,
) -> WSBFormat:
    """
    Build WSB format from torch.sparse CSR tensor.

    Algorithm:
    1. Divide nodes into row windows of 16 rows each
    2. For each row window:
       a. Collect all unique column indices from the 16 rows
       b. Sort and partition into TCBs of 8 columns each
       c. For each TCB, build bitmap and weight array

    Args:
        adj: sparse СSR tensor of adjacency matrix
        dtype: Weight dtype (e.g. float16 for tensor cores)

    Returns:
        WSBFormat with all tensors ready for CUDA kernel
    """
    N = adj.shape[0]
    assert adj.shape[0] == adj.shape[1], "Adjacency must be square"

    indptr = adj.crow_indices()
    indices = adj.col_indices()
    weights = adj.values()


    num_row_windows = (N + ROW_WINDOW_SIZE - 1) // ROW_WINDOW_SIZE
    num_edges = len(indices)


    tcb_row_offset = [0]
    all_col_idx = []  # 8 columns per TCB
    all_bitmaps = []  # 2 uint64 per TCB
    all_weights = []  # 128 floats per TCB

    # process each row window
    for rw in range(num_row_windows):
        row_start = rw * ROW_WINDOW_SIZE
        row_end = min(row_start + ROW_WINDOW_SIZE, N)
        num_rows_in_window = row_end - row_start

        # collect all (local_row, col, weight) for this row window
        edges_in_window = []
        for local_row in range(num_rows_in_window):
            global_row = row_start + local_row
            for idx in range(indptr[global_row], indptr[global_row + 1]):
                col = indices[idx].item()
                w = weights[idx].item()
                # print(f"{rw=} {idx=} {global_row=} {col=} {w=}")
                edges_in_window.append((local_row, col, w))

        if len(edges_in_window) == 0:
            tcb_row_offset.append(tcb_row_offset[-1])
            continue

        # get unique columns and sort them
        unique_cols = sorted(set(e[1] for e in edges_in_window))
        num_unique_cols = len(unique_cols)

        # column -> local index mapping
        col_to_local = {c: i for i, c in enumerate(unique_cols)}

        # edge lookup: (local_row, local_col) -> weight
        edge_map = {}
        for local_row, col, w in edges_in_window:
            local_col = col_to_local[col]
            edge_map[(local_row, local_col)] = w

        # number of TCBs for this row window
        num_tcbs_in_rw = (num_unique_cols + TCB_WIDTH - 1) // TCB_WIDTH

        # process each TCB
        for tcb_idx in range(num_tcbs_in_rw):
            col_start = tcb_idx * TCB_WIDTH
            col_end = min(col_start + TCB_WIDTH, num_unique_cols)

            # column indices for this TCB (pad with 0 if fewer than 8)
            tcb_cols = unique_cols[col_start:col_end]
            while len(tcb_cols) < TCB_WIDTH:
                tcb_cols.append(0)  # padding
            all_col_idx.extend(tcb_cols)

            # build bitmap and weights for this TCB
            # bitmap layout: bits 0-63 for rows 0-7 (first uint64)
            #                bits 0-63 for rows 8-15 (second uint64)
            # within each uint64: bit = row * 8 + col_in_tcb
            bitmap_lo = np.uint64(0)  # Rows 0-7
            bitmap_hi = np.uint64(0)  # Rows 8-15
            tcb_weights = np.zeros(TCB_SIZE, dtype=np.float32)

            for local_row in range(ROW_WINDOW_SIZE):
                for local_col_in_tcb in range(TCB_WIDTH):
                    global_local_col = col_start + local_col_in_tcb

                    if global_local_col >= num_unique_cols:
                        continue

                    key = (local_row, global_local_col)
                    if key in edge_map:
                        # set bit in bitmap
                        bit_pos = (local_row % 8) * TCB_WIDTH + local_col_in_tcb
                        if local_row < 8:
                            bitmap_lo |= np.uint64(1) << np.uint64(bit_pos)
                        else:
                            bitmap_hi |= np.uint64(1) << np.uint64(bit_pos)

                        # store weight (row-major within TCB)
                        weight_idx = local_row * TCB_WIDTH + local_col_in_tcb
                        tcb_weights[weight_idx] = edge_map[key]

            all_bitmaps.extend([bitmap_lo, bitmap_hi])
            all_weights.extend(tcb_weights.tolist())

        tcb_row_offset.append(tcb_row_offset[-1] + num_tcbs_in_rw)

    # convert to tensors
    num_tcbs = tcb_row_offset[-1]
    bitmap_array = np.array(all_bitmaps, dtype=np.uint64)
    bitmap_tensor = torch.from_numpy(bitmap_array.view(np.int64)).clone()

    return WSBFormat(
        tcb_row_offset=torch.tensor(tcb_row_offset, dtype=torch.int32),
        col_idx=torch.tensor(all_col_idx, dtype=torch.int32),
        bitmap=bitmap_tensor,
        weights=torch.tensor(all_weights, dtype=dtype),
        num_nodes=N,
        num_edges=num_edges,
        num_row_windows=num_row_windows,
        num_tcbs=num_tcbs,
    )


In [14]:
ones = torch.ones((16, 16))
zeros = torch.zeros((16, 16))

adj = torch.cat(
    [
        torch.cat([ones, zeros], 1),
        torch.cat([zeros, ones], 1),
    ],
    0,
)

adj[1, 17] = 1

edge_index = adj.nonzero().T
adjacency_sparse = normalize_adj(edge_index, num_nodes=adj.shape[0], how="both", add_self_loops=False)
adj_csr = adjacency_sparse.to_sparse_csr()


In [15]:
from ogb.nodeproppred import DglNodePropPredDataset

In [16]:
import dgl

In [17]:
dataset = DglNodePropPredDataset(name="ogbn-products", root="/home/fvelikon/projects/cuda_exp/data")
g, _ = dataset[0]
g = dgl.add_self_loop(g)

In [18]:
edge_index = torch.stack(g.edges())


In [ ]:
adjacency_sparse = normalize_adj(edge_index, num_nodes=g.num_nodes(), how="both", add_self_loops=False)
adj_csr = adjacency_sparse.to_sparse_csr()

In [20]:
wsb = build_wsb_format(adj_csr)

In [21]:
wsb

WSBFormat(nodes=2449029, edges=126167053, row_windows=153065, tcbs=15820655, memory=4697354.9 KB)

In [50]:
dir(wsb)

['__add__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__contains__',
 '__delattr__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getitem__',
 '__getnewargs__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__len__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__mul__',
 '__ne__',
 '__new__',
 '__orig_bases__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__rmul__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '_asdict',
 '_field_defaults',
 '_fields',
 '_make',
 '_replace',
 'bitmap',
 'col_idx',
 'count',
 'cuda',
 'index',
 'memory_bytes',
 'num_edges',
 'num_nodes',
 'num_row_windows',
 'num_tcbs',
 'tcb_row_offset',
 'to',
 'to_dense',
 'weights']

In [51]:
wsb.bitmap.shape

torch.Size([31641310])

In [56]:
wsb.num_tcbs

15820655

In [54]:
wsb.num_tcbs

15820655

In [ ]:
g.edge

In [57]:
a = adj_csr.to("cuda")

In [35]:
a = wsb.cuda()

In [27]:
a = g.to("cuda")

In [38]:
del a

NameError: name 'a' is not defined

In [37]:
torch.cuda.empty_cache()

In [22]:
wsb.weights.shape

torch.Size([2025043840])

In [17]:
adj.shape

torch.Size([32, 32])

In [18]:
wsb.num_tcbs

5

In [19]:
torch.allclose(wsb.to_dense(), adjacency_sparse.to_dense(), rtol=1e-3, atol=1e-3)

True

In [20]:
import networkx as nx

In [21]:
N = [16, 32, 64, 58, 41]

for n in N:
    for p in [0.1, 0.2, 0.5, 0.9]:
        for _ in range(10):
            g = nx.gnp_random_graph(n, p)
            adj_g = torch.from_numpy(nx.to_numpy_array(g))
            edge_index = adj_g.nonzero().T

            adjacency_sparse = normalize_adj(edge_index, num_nodes=adj_g.shape[0], how="both", add_self_loops=False)
            adj_csr = adjacency_sparse.to_sparse_csr()

            wsb = build_wsb_format(adj_csr)
            assert torch.allclose(wsb.to_dense(), adjacency_sparse.to_dense(), rtol=1e-3, atol=1e-3)